<center>

# **Build a Direct Lake semantic model from a Lakehouse**

</center>

### Purpose
Bootstraps a Direct Lake semantic model from a Lakehouse using **[Semantic Link Labs](https://github.com/microsoft/semantic-link-labs)**.

The notebook auto-classifies tables by naming convention (`dim_` / `fact_`), builds relationships from key-suffix columns (`_id` / `_key`), adds a few generic measures, and finally adds a **Time Intelligence calculation group**.

Run it as-is over a gold lakehouse that follows the prefix convention - no per-model code required.

> Requires `semantic-link-labs` (install in the notebook: `%pip install semantic-link-labs`).

### Configuration
Point the notebook at a Lakehouse and (optionally) override the naming patterns. The lakehouse / workspace can be supplied by name or ID; leave a value as `None` to default to the current notebook context.

In [ ]:
# --- Source Lakehouse (where the gold tables live) ---
lakehouse = None              # Lakehouse name or ID. None = current notebook lakehouse.
lakehouse_workspace = None    # Workspace name or ID for the lakehouse. None = current workspace.

# --- Target semantic model ---
dataset_name = "Sales - Auto Generated"
dataset_workspace = None      # Workspace to create the model in. None = current workspace.

# --- Naming conventions used to classify tables and build relationships ---
dim_prefix = "dim_"           # Tables starting with this prefix are dimensions
fact_prefix = "fact_"         # Tables starting with this prefix are facts
key_suffixes = ["_id", "_key"]  # Columns ending with any of these are relationship keys

# --- Behavior switches ---
overwrite_existing = True     # If True, drop and recreate the model if it already exists
add_time_intelligence = True  # Adds the Time Intelligence calc group on top of fact measures
date_table_hint = "dim_date"  # Dimension treated as the Date table for time intelligence (lower-cased match)

### Helpers
Small utilities for classifying tables, finding relationship keys, and adding measures via the TOM wrapper.

In [ ]:
import sempy_labs as labs
from sempy_labs import lakehouse as lh
from sempy_labs import directlake
from sempy_labs.tom import connect_semantic_model


def classify_tables(table_names, dim_prefix, fact_prefix):
    """Split table names into (dimensions, facts, ignored) by prefix - case-insensitive."""
    dims, facts, other = [], [], []
    for t in table_names:
        n = t.lower()
        if n.startswith(dim_prefix.lower()):
            dims.append(t)
        elif n.startswith(fact_prefix.lower()):
            facts.append(t)
        else:
            other.append(t)
    return dims, facts, other


def is_key_column(col_name, key_suffixes):
    n = col_name.lower()
    return any(n.endswith(s.lower()) for s in key_suffixes)


def dim_primary_key(dim_name, dim_columns, key_suffixes):
    """Pick the relationship key on a dimension. Prefer '<entity>_id' / '<entity>_key' that matches the table stem."""
    stem = dim_name.lower().removeprefix(dim_prefix.lower())
    candidates = [c for c in dim_columns if is_key_column(c, key_suffixes)]
    # Best match: stem + suffix (e.g. dim_customer -> customer_id)
    for c in candidates:
        if c.lower().startswith(stem) or c.lower() == f"{stem}_id" or c.lower() == f"{stem}_key":
            return c
    return candidates[0] if candidates else None

### Inspect the Lakehouse
List the tables and apply the prefix-based classification. The output is what the rest of the notebook will model.

In [ ]:
tables_df = lh.get_lakehouse_tables(lakehouse=lakehouse, workspace=lakehouse_workspace)
table_names = tables_df["Table Name"].tolist()

dims, facts, ignored = classify_tables(table_names, dim_prefix, fact_prefix)

print(f"Found {len(table_names)} table(s) in lakehouse.")
print(f"  Dimensions ({len(dims)}): {dims}")
print(f"  Facts      ({len(facts)}): {facts}")
print(f"  Ignored    ({len(ignored)}): {ignored}")

### Bootstrap the Direct Lake semantic model
Uses `generate_direct_lake_semantic_model` to create the model, mapping the lakehouse Delta tables straight onto the model. Only classified dims and facts are included.

In [ ]:
model_tables = dims + facts

if not model_tables:
    raise RuntimeError(
        f"No tables matched the prefixes '{dim_prefix}' / '{fact_prefix}'. "
        "Adjust the prefixes in the configuration or check the lakehouse contents."
    )

directlake.generate_direct_lake_semantic_model(
    dataset=dataset_name,
    workspace=dataset_workspace,
    lakehouse=lakehouse,
    lakehouse_workspace=lakehouse_workspace,
    lakehouse_tables=model_tables,
    overwrite=overwrite_existing,
    refresh=True,
)

print(f"Direct Lake semantic model '{dataset_name}' created with {len(model_tables)} table(s).")

### Add relationships from key-suffix columns
Walks every fact table, finds columns matching the configured key suffixes, and matches them to a dimension by name (`dim_<entity>` <-> `<entity>_id` or `<entity>_key`).

In [ ]:
with connect_semantic_model(dataset=dataset_name, workspace=dataset_workspace, readonly=False) as tom:
    # Snapshot column lists per table (for quick lookup)
    table_columns = {t.Name: [c.Name for c in t.Columns] for t in tom.model.Tables}

    # Pre-resolve a primary key per dimension
    dim_keys = {}
    for d in dims:
        if d in table_columns:
            pk = dim_primary_key(d, table_columns[d], key_suffixes)
            if pk:
                dim_keys[d] = pk

    created, skipped = [], []
    for f in facts:
        for fk in table_columns.get(f, []):
            if not is_key_column(fk, key_suffixes):
                continue
            # Map fact key 'customer_id' -> dim 'dim_customer'
            stem = fk.lower()
            for s in key_suffixes:
                if stem.endswith(s.lower()):
                    stem = stem[: -len(s)]
                    break
            target_dim = next((d for d in dims if d.lower() == f"{dim_prefix.lower()}{stem}"), None)
            if not target_dim or target_dim not in dim_keys:
                skipped.append((f, fk))
                continue
            try:
                tom.add_relationship(
                    from_table=f,
                    from_column=fk,
                    to_table=target_dim,
                    to_column=dim_keys[target_dim],
                    from_cardinality="Many",
                    to_cardinality="One",
                )
                created.append((f, fk, target_dim, dim_keys[target_dim]))
            except Exception as ex:
                skipped.append((f, fk, str(ex)))

    print(f"Created {len(created)} relationship(s):")
    for r in created:
        print(f"  {r[0]}[{r[1]}] -> {r[2]}[{r[3]}]")
    if skipped:
        print(f"\nSkipped {len(skipped)}:")
        for s in skipped:
            print(f"  {s}")

### Add generic measures on fact tables
Adds a row-count measure per fact table plus a sum measure for any numeric column that is *not* a key. This is intentionally generic - tweak the rules for your gold model.

In [ ]:
NUMERIC_TYPES = {"Int64", "Decimal", "Double"}

with connect_semantic_model(dataset=dataset_name, workspace=dataset_workspace, readonly=False) as tom:
    for fact in facts:
        table = next((t for t in tom.model.Tables if t.Name == fact), None)
        if table is None:
            continue

        # 1. Row count measure
        tom.add_measure(
            table_name=fact,
            measure_name=f"# {fact}",
            expression=f"COUNTROWS('{fact}')",
            format_string="#,0",
        )

        # 2. Sum of every non-key numeric column
        for col in table.Columns:
            if str(col.DataType) not in NUMERIC_TYPES:
                continue
            if is_key_column(col.Name, key_suffixes):
                continue
            measure_name = f"Total {col.Name.replace('_', ' ').title()}"
            tom.add_measure(
                table_name=fact,
                measure_name=measure_name,
                expression=f"SUM('{fact}'[{col.Name}])",
                format_string="#,0.00",
            )

    print("Generic measures added.")

### Add a Time Intelligence calculation group
A reusable calc group that lets every measure be evaluated as YTD / MTD / QTD / PY / PY YTD / YoY / YoY %. Requires the date dimension to be marked as a date table - the cell does that based on `date_table_hint`.

In [ ]:
if add_time_intelligence:
    with connect_semantic_model(dataset=dataset_name, workspace=dataset_workspace, readonly=False) as tom:
        # Find the date dimension (case-insensitive match against hint)
        date_table = next((t.Name for t in tom.model.Tables if t.Name.lower() == date_table_hint.lower()), None)
        if not date_table:
            print(f"Date dimension '{date_table_hint}' not found - skipping Time Intelligence calc group.")
        else:
            # Mark the date column as the date table key
            date_col = next(
                (c.Name for c in tom.model.Tables[date_table].Columns
                 if c.Name.lower() in {"date", "datekey", "date_key", "date_id"}),
                None,
            )
            if date_col:
                tom.mark_as_date_table(table_name=date_table, column_name=date_col)

            tom.add_calculation_group(name="Time Intelligence", precedence=1)

            items = [
                ("Current",  "SELECTEDMEASURE()", 0),
                ("YTD",      f"CALCULATE(SELECTEDMEASURE(), DATESYTD('{date_table}'[{date_col}]))", 10),
                ("QTD",      f"CALCULATE(SELECTEDMEASURE(), DATESQTD('{date_table}'[{date_col}]))", 20),
                ("MTD",      f"CALCULATE(SELECTEDMEASURE(), DATESMTD('{date_table}'[{date_col}]))", 30),
                ("PY",       f"CALCULATE(SELECTEDMEASURE(), SAMEPERIODLASTYEAR('{date_table}'[{date_col}]))", 40),
                ("PY YTD",   f"CALCULATE(SELECTEDMEASURE(), SAMEPERIODLASTYEAR(DATESYTD('{date_table}'[{date_col}])))", 50),
                ("YoY",      "SELECTEDMEASURE() - CALCULATE(SELECTEDMEASURE(), 'Time Intelligence'[Time Intelligence] = \"PY\")", 60),
                ("YoY %",    "DIVIDE("
                              "CALCULATE(SELECTEDMEASURE(), 'Time Intelligence'[Time Intelligence] = \"YoY\"),"
                              "CALCULATE(SELECTEDMEASURE(), 'Time Intelligence'[Time Intelligence] = \"PY\"))", 70),
            ]
            for name, expr, ordinal in items:
                tom.add_calculation_item(
                    table_name="Time Intelligence",
                    calculation_item_name=name,
                    expression=expr,
                    ordinal=ordinal,
                )

            print(f"Time Intelligence calc group added (date table: '{date_table}'[{date_col}]).")
else:
    print("Time intelligence skipped (add_time_intelligence = False).")

### Done
The Direct Lake model now has:
- Tables for every dim/fact in the lakehouse that matched the prefix rules
- Relationships derived from `_id` / `_key` matching
- A row-count measure plus sum measures on every non-key numeric column
- A reusable Time Intelligence calculation group

Open the model in Power BI or pair it with the autogen-report notebook to spin up a report on top.